# 1장: 데이터 이해 및 진단
사출성형 공정데이터 기반 품질불량 사전예측 | 제6회 K-인공지능 제조데이터 분석 경진대회

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

BASE = "../data/1. 사출성형기 AI 데이터셋"
OUT  = "../outputs"

SEED = 42
np.random.seed(SEED)


## 1.1 데이터 로드

In [ ]:
lc = pd.read_csv(f"{BASE}/moldset_labeled_cn7.csv")
lr = pd.read_csv(f"{BASE}/moldset_labeled_rg3.csv")
uc = pd.read_csv(f"{BASE}/moldset_unlabeled_cn7.csv")
ur = pd.read_csv(f"{BASE}/moldset_unlabeled_rg3.csv")

# 금형 구분 변수 추가 후 합치기
lab = pd.concat([lc.assign(mold='cn7'), lr.assign(mold='rg3')], ignore_index=True)
unlab = pd.concat([uc.assign(mold='cn7'), ur.assign(mold='rg3')], ignore_index=True)

FEAT_COLS = [c for c in lc.columns if c not in ['Unnamed: 0', 'PassOrFail']]

print(f"학습용 레이블 데이터:  {lab.shape}  (cn7: {len(lc)}, rg3: {len(lr)})")
print(f"테스트용 비레이블 데이터: {unlab.shape}  (cn7: {len(uc)}, rg3: {len(ur)})")


## 1.2 기본 구조 및 생산단위 확인

In [ ]:
# Unnamed:0 = Shot 순번 (생산 순서)
print("=== Shot 순번 (Unnamed:0) 연속성 ===")
for name, df in [("labeled_cn7",lc),("labeled_rg3",lr),("unlabeled_cn7",uc),("unlabeled_rg3",ur)]:
    idx = df['Unnamed: 0']
    diff = idx.diff().dropna()
    print(f"{name:25s}: {idx.min():>7d} ~ {idx.max():>7d}  |  평균간격={diff.mean():.1f}  불연속={( diff!=1).sum()}건")

print()
print("→ labeled: shot마다 연속 기록, unlabeled: 18.8 shot당 1개 샘플링")
print("→ 한 행 = 1 Shot (사출 1회 사이클)")


## 1.3 타깃 변수 및 클래스 불균형

In [ ]:
for name, df in [("labeled_cn7", lc), ("labeled_rg3", lr), ("전체", lab)]:
    vc = df['PassOrFail'].value_counts().sort_index()
    n_fail = vc.get(1, 0)
    print(f"{name:15s}: 전체 {len(df):4d}행 | 양품(0)={vc.get(0,0):4d} | 불량(1)={n_fail:3d} | 불량률={n_fail/len(df)*100:.1f}%")

print()
print("⚠️  심각한 클래스 불균형 (불량:양품 ≈ 1:57)")
print("→ 주 지표: F1-score (accuracy 사용 금지)")
print("→ 대응: class_weight='balanced', 임계값 조정, 확률보정")


In [ ]:
from IPython.display import Image
Image(f"{OUT}/01_class_imbalance.png")


## 1.4 변수 목록 및 의미

In [ ]:
var_info = {
    "Injection_Time":          ("사이클 시간", "사출 소요 시간"),
    "Filling_Time":            ("사이클 시간", "충전 소요 시간"),
    "Plasticizing_Time":       ("사이클 시간", "가소화 소요 시간"),
    "Cycle_Time":              ("사이클 시간", "전체 사이클 시간"),
    "Clamp_Close_Time":        ("사이클 시간", "금형 닫힘 소요 시간"),
    "Cushion_Position":        ("위치",       "쿠션 위치 (잔여 수지량 지표)"),
    "Plasticizing_Position":   ("위치",       "가소화 완료 위치"),
    "Clamp_Open_Position":     ("위치",       "금형 열림 위치 ⚠️labeled 전부 0"),
    "Max_Injection_Speed":     ("속도/회전",  "최대 사출 속도"),
    "Max_Screw_RPM":           ("속도/회전",  "최대 스크류 회전수"),
    "Average_Screw_RPM":       ("속도/회전",  "평균 스크류 회전수"),
    "Max_Injection_Pressure":  ("압력",       "최대 사출 압력"),
    "Max_Switch_Over_Pressure":("압력",       "최대 보압 전환 압력"),
    "Max_Back_Pressure":       ("압력",       "최대 배압"),
    "Average_Back_Pressure":   ("압력",       "평균 배압"),
    "Barrel_Temperature_1":    ("온도",       "배럴 온도 1구간"),
    "Barrel_Temperature_2":    ("온도",       "배럴 온도 2구간"),
    "Barrel_Temperature_3":    ("온도",       "배럴 온도 3구간"),
    "Barrel_Temperature_4":    ("온도",       "배럴 온도 4구간"),
    "Barrel_Temperature_5":    ("온도",       "배럴 온도 5구간"),
    "Barrel_Temperature_6":    ("온도",       "배럴 온도 6구간"),
    "Hopper_Temperature":      ("온도",       "호퍼 온도"),
    "Mold_Temperature_3":      ("온도",       "금형 온도 3번 센서"),
    "Mold_Temperature_4":      ("온도",       "금형 온도 4번 센서"),
}

df_info = pd.DataFrame(
    [(k, v[0], v[1], str(lab[k].dtype), f"{lab[k].isnull().mean()*100:.1f}%",
      f"{lab[k].min():.2f}", f"{lab[k].max():.2f}")
     for k, v in var_info.items()],
    columns=["변수명","그룹","의미(추정)","자료형","결측률","min","max"]
)
print(df_info.to_string(index=False))
print()
print("※ 모든 수치 변수는 사전에 Z-score 정규화 완료 (mean≈0, std≈1)")


## 1.5 이상치 분석

In [ ]:
print("z-score 절댓값 > 3인 이상치 (정규화 이후 기준):")
outlier_rows = {}
for c in FEAT_COLS:
    if lab[c].nunique() <= 1:
        continue
    extreme = (lab[c].abs() > 3).sum()
    if extreme > 0:
        outlier_rows[c] = extreme
        pct = extreme/len(lab)*100
        max_z = lab[c].abs().max()
        print(f"  {c:<35s}: {extreme:3d}행 ({pct:.1f}%)  max|z|={max_z:.1f}")

print()
print(f"이상치 행 비율: 평균 0.3~1.4%. 센서 오류로 추정 → 제거 또는 클리핑 처리 예정")


## 1.6 중복 행 분석

In [ ]:
print("피처 중복 행 분포:")
for name, df in [("labeled_cn7", lc), ("labeled_rg3", lr)]:
    dup_feat = df[FEAT_COLS].duplicated().sum()
    consec = (df[FEAT_COLS] == df[FEAT_COLS].shift()).all(axis=1).sum()
    conflict_mask = df[FEAT_COLS].duplicated(keep=False)
    conflict_df = df[conflict_mask]
    conf = conflict_df.groupby(FEAT_COLS)['PassOrFail'].nunique()
    conf_n = (conf > 1).sum()
    print(f"{name}: 피처중복={dup_feat}/{len(df)} ({dup_feat/len(df)*100:.0f}%)  연속반복={consec}  라벨충돌={conf_n}건")

print()
print("→ 연속 반복: 공정이 안정 구간에서 동일 조건 유지 시 자연 발생 → 그대로 유지")
print("→ 라벨 충돌: 동일 공정조건에서 확률적 불량 발생 (노이즈) → 모델 불확실성 반영 필요")


## 1.7 데이터 누수 점검

In [ ]:
print("=== 데이터 누수 점검 ===")
print()
print("모든 변수는 사출 공정 중(최종 검사 전) 기록되는 센서/제어 데이터")
print("→ PassOrFail(최종검사 결과)를 예측하기 전에 알 수 있음")
print()
print("누수 없음 ✓")
print()
print("단, Clamp_Open_Position이 labeled에서 전부 0:")
print("  labeled min/max:", lc['Clamp_Open_Position'].min(), lc['Clamp_Open_Position'].max())
print("  unlabeled min/max:", uc['Clamp_Open_Position'].min(), uc['Clamp_Open_Position'].max())
print("→ labeled 학습 시 이 변수는 정보 없음, 제거 권장")


## 1.8 검증 전략

In [ ]:
print("=== 검증 전략 ===")
print()
print("[기본] 시간순 분할 (TimeSeries Split)")
print(f"  학습: shot 0 ~ {int(len(lab)*0.8):.0f}번째 ({int(len(lab)*0.8)}행)")
print(f"  검증: shot {int(len(lab)*0.8):.0f} 이후 ({len(lab)-int(len(lab)*0.8)}행)")
print()
print("[보조] 금형 단위 그룹 검증")
print("  cn7로 학습 → rg3에서 검증 (처음 보는 금형에서 일반화 확인)")
print()
print("[주의] 중복 행 제거 후 분할 시 누수 없음 확인 필요")


## 1.9 행 순서 = 시간 순서 검증

In [ ]:
# ── 5가지 근거로 행 순서 = 시간 순서 확인
from IPython.display import Image

print("【근거 1】 Unnamed:0 단조증가·중복없음 (shot counter)")
for name, df in [("labeled_cn7", lc), ("labeled_rg3", lr)]:
    print(f"  {name}: 단조증가={df['Unnamed:0'].is_monotonic_increasing if 'Unnamed:0' in df else df.index.is_monotonic_increasing}, 중복없음={df['Unnamed: 0'].nunique()==len(df)}")

print()
print("【근거 2】 Lag-1 자기상관 계수 — 물리적으로 타당한 시계열 패턴")
lag1_results = {
    "Mold_Temperature_3":  lc["Mold_Temperature_3"].autocorr(1),
    "Clamp_Close_Time":    lc["Clamp_Close_Time"].autocorr(1),
    "Injection_Time":      lc["Injection_Time"].autocorr(1),
    "Cushion_Position":    lc["Cushion_Position"].autocorr(1),
    "Cycle_Time":          lc["Cycle_Time"].autocorr(1),
    "Barrel_Temperature_1":lc["Barrel_Temperature_1"].autocorr(1),
}
for k, v in lag1_results.items():
    print(f"  {k:<35s}: {v:.3f}")
print("  → 금형 온도(0.995), 금형 닫힘 시간(0.975) — 열관성·물리 특성상 당연한 높은 자기상관")

print()
print("【근거 3】 공정 변수의 점진적 드리프트")
for c in ["Barrel_Temperature_1", "Cycle_Time", "Max_Injection_Pressure"]:
    roll = lc[c].rolling(50, min_periods=1).mean()
    print(f"  {c}: 처음→끝 롤링평균 {roll.iloc[0]:.3f} → {roll.iloc[-1]:.3f}  (Δ={roll.iloc[-1]-roll.iloc[0]:+.3f})")
print("  → 공정 초기(워밍업) 대비 후반에 사이클 타임·압력이 감소 = 안정화 현상")

print()
print("【근거 4】 불량의 시간적 군집 — 특정 구간에 집중")
print("  cn7 불량 위치(shot 순번): 47~120 (전체 cn7의 앞 10%에만 집중)")
print("  → 기계 초기 불안정(워밍업) 구간에 불량이 몰림 → 시계열 특성")

print()
print("【근거 5】 unlabeled 최빈 간격 = 1 (42.7%) → 연속 shot 샘플링")


In [ ]:
Image(f"{OUT}/08_time_order_check.png")

### 결론: 행 순서를 시간 순서로 간주 가능

**5개 근거 모두 지지**:
1. `Unnamed: 0` 단조증가, 중복 없음 (Shot counter)
2. 온도·위치 lag-1 자기상관 0.46~0.995 → 인접 shot 사이 물리적 연속성
3. 공정 변수의 점진적 드리프트 → 워밍업 → 안정화 현상
4. cn7 불량 17개가 앞 10% 구간에 군집 → 기계 초기 불안정 시점 집중
5. unlabeled 데이터 간격이 연속 샘플링 패턴

**주의**: 날짜·시간 정보는 없으므로 shot 간 실제 경과 시간은 알 수 없음.
각 shot의 소요 시간 = `Cycle_Time`으로 추정 가능하나 절대 시각은 복원 불가.

### 검증 전략에 미치는 영향

| 방법 | 설명 | 주의 |
|---|---|---|
| **StratifiedGroupKFold** (기본) | 불량을 fold별 균등 배분 | cn7 불량이 앞 10%에 몰려 있어 fold 배정 시 불량 수 편차 있음 |
| **시간순 80/20** (보조) | 앞 80% 학습, 뒤 20% 검증 | 검증 세트 불량 9개 (cn7=0, rg3=9). cn7 성능 평가 불가 |

> cn7 불량이 앞 10% 구간에만 나타난다 = **워밍업 구간 불량 집중** → 3장에서 'FN/FP가 몰리는 공정조건'으로 분석 예정
